# 184. Department Highest Salary

## Problem
Employee(id,name,salary,departmentId), Department(id,name): return Department, Employee, Salary for every employee at their department maximum. Keep ties; any order.

[Doocs problem reference](https://github.com/doocs/leetcode/blob/main/solution/0100-0199/0184.Department%20Highest%20Salary/README_EN.md) · [LeetCode](https://leetcode.com/problems/department-highest-salary/)

[Community solution index (Silversaya, 2023)](https://leetcode.cn/discuss/post/3573106/zhe-shi-pandas30tian-ji-hua-de-ti-jie-qu-v0wv/) · [LeetCode discussions](https://leetcode.com/problems/department-highest-salary/solutions/) · [Individual community solution](https://leetcode.cn/problems/department-highest-salary/solutions/2373267/184-bu-men-gong-zi-zui-gao-de-yuan-gong-3dctk/)

Community links are for further reading; the implementations below are independently written.

## Practice first
Run the data cell and try the blank cell before reading solutions.

In [ ]:
import pandas as pd
from pandas.testing import assert_frame_equal
from decimal import Decimal, ROUND_HALF_UP

def percent(n,d):
    return float((Decimal(int(n))*100/Decimal(int(d))).quantize(Decimal('0.01'),rounding=ROUND_HALF_UP)) if d else None

employee = pd.DataFrame([[1,"A",90,1],[2,"B",90,1],[3,"C",80,2],[4,"D",10,1]], columns=["id","name","salary","departmentId"])
department = pd.DataFrame([[1,"IT"],[2,"Sales"],[3,"Empty"]], columns=["id","name"])
expected = pd.DataFrame([["IT","A",90],["IT","B",90],["Sales","C",80]], columns=["Department","Employee","Salary"])
display(employee)
display(department)
display(expected)

In [ ]:
# Your solution here
# result = ...

## Solutions

### Solution 1 — Transform
transform 保留資料粒度，idxmax 會漏掉並列。

In [ ]:
def solution_1(employee, department):
    m = employee.merge(department,left_on='departmentId',right_on='id',suffixes=('_emp','_dept'))
    top = m.loc[m.salary.eq(m.groupby('departmentId').salary.transform('max'))]
    return top.rename(columns={'name_dept':'Department','name_emp':'Employee','salary':'Salary'})[['Department','Employee','Salary']]

result_1 = solution_1(employee.copy(), department.copy())
display(result_1)

### Solution 2 — Aggregate merge
先彙總再 join；必須使用兩個鍵。

In [ ]:
def solution_2(employee, department):
    g = employee.groupby('departmentId',as_index=False).salary.max()
    m = employee.merge(g,on=['departmentId','salary']).merge(department,left_on='departmentId',right_on='id',suffixes=('_emp','_dept'))
    return m.rename(columns={'name_dept':'Department','name_emp':'Employee','salary':'Salary'})[['Department','Employee','Salary']]

result_2 = solution_2(employee.copy(), department.copy())
display(result_2)

### Solution 3 — Group rank
延伸至各組 Top N。

In [ ]:
def solution_3(employee, department):
    top = employee.loc[employee.groupby('departmentId').salary.rank(method='dense',ascending=False).eq(1)]
    m = top.merge(department,left_on='departmentId',right_on='id',suffixes=('_emp','_dept'))
    return m.rename(columns={'name_dept':'Department','name_emp':'Employee','salary':'Salary'})[['Department','Employee','Salary']]

result_3 = solution_3(employee.copy(), department.copy())
display(result_3)

## Check all solutions
Rows are normalized only for the example comparison; preserve required ordering in the solutions themselves.

In [ ]:
def normalize(df):
    return df.sort_values(list(df.columns),na_position='last').reset_index(drop=True)
assert_frame_equal(normalize(result_1), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_2), normalize(expected), check_dtype=False)
assert_frame_equal(normalize(result_3), normalize(expected), check_dtype=False)
print('All example checks passed')

## Review
Compare readability, intermediate data size and handling of duplicate values, missing matches and empty groups.